<a href="https://colab.research.google.com/github/Jake-Song/KG-agent/blob/main/notebooks/quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# KG Agent quickstart

Use a knowledge graph as an agent's persistent world model, planner state, and hallucination gate. This walkthrough is deterministic, uses only Python's standard library, and takes about two minutes.


## 1. Load the project

A Colab runtime starts empty, so clone the repository and import it directly. Re-running this cell is safe.


In [ ]:
from pathlib import Path
import subprocess
import sys

repo_dir = Path("/content/KG-agent")
if not repo_dir.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/Jake-Song/KG-agent.git", str(repo_dir)],
        check=True,
    )

if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

print(f"KG Agent ready from {repo_dir}")


## 2. Build one shared world model

Nodes are entities; typed relations describe facts and dependencies. Edge metadata records where each belief came from and how confident it is.


In [ ]:
from kg_agent import (
    Claim, Goal, KGAgent, KnowledgeGraph, ScriptedLLM,
    default_ontology, plan_for, verify,
)

kg = KnowledgeGraph(ontology=default_ontology())

for name, kind in [
    ("Hypothesis_A", "Hypothesis"),
    ("Measurement_B", "Measurement"),
    ("Instrument_C", "Instrument"),
    ("Lab_D", "Lab"),
    ("Protein_A", "Protein"),
    ("Protein_B", "Protein"),
]:
    kg.add_node(name, kind)

kg.assert_edge("Hypothesis_A", "requires", "Measurement_B", source="protocol", strict=True)
kg.assert_edge("Measurement_B", "requires", "Instrument_C", source="protocol", strict=True)
kg.assert_edge("Instrument_C", "requires", "Lab_D", source="protocol", strict=True)
kg.assert_edge("Protein_A", "activates", "Protein_B", source="assay", confidence=0.95, strict=True)

print(kg.context_for("Hypothesis_A", hops=3))


## 3. Plan from graph dependencies

The planner backward-chains from the goal. Prerequisites become ordered stages; completed nodes are automatically pruned on the next plan.


In [ ]:
goal = Goal("Hypothesis_A", "validate Hypothesis A")
plan = plan_for(kg, goal)
print(plan.render())


## 4. Check generated claims before believing them

Verification distinguishes graph-supported statements from contradictions and unknowns. Here, `inhibits` conflicts with the stored `activates` relation.


In [ ]:
claims_to_check = [
    Claim.parse("Protein_A activates Protein_B"),
    Claim.parse("Protein_A inhibits Protein_B"),
    Claim.parse("Protein_B activates Protein_A"),
]

for claim in claims_to_check:
    verdict = verify(kg, claim)
    print(f"{verdict.status.value:12} | {claim}")
    print(f"               {verdict.explanation}")


## 5. Run the agent loop

`ScriptedLLM` makes this example reproducible. Replace it later with `OpenRouterLLM` or any object implementing the three-method `LLM` protocol. Action handlers change graph state; replanning then advances to the next prerequisite.


In [ ]:
agent = KGAgent(kg, ScriptedLLM())

@agent.action("secure_lab_access")
def secure_lab_access(agent, step):
    return f"access granted for {step.node}"

@agent.action("acquire_instrument")
def acquire_instrument(agent, step):
    return f"acquired {step.node}"

@agent.action("run_measurement")
def run_measurement(agent, step):
    return f"completed {step.node}"

@agent.action("evaluate_hypothesis")
def evaluate_hypothesis(agent, step):
    return f"validated {step.node}"

run = agent.run(goal)
print(run.render())


In [ ]:
print("Final plan after graph updates:")
print(agent.plan(goal).render())
print(f"\nGraph revision: {kg.revision}; model calls: {len(agent.llm.calls)}; retained message history: 0")


## Next steps

- Save state with `kg.save("/content/world.json")`; restore it with `KnowledgeGraph.load(...)`.
- For a live model, add `OPENROUTER_API_KEY` to Colab Secrets and follow the repository's OpenRouter example. Generated claims still pass through the same verification gate.
- Define an `Ontology` with your own entity types, relation constraints, inverse relations, and dependency predicates.

See the [project README](https://github.com/Jake-Song/KG-agent#readme) for the complete API and architecture.
